# Transfer learning sem TS9: PaSST e HTS-AT

Consulta da ablação com 12 classes nos quatro cenários GUITAR-FX-DIST. Encoders congelados; cabeças novas, cinco seeds finais por modelo. O experimento original com 13 classes permanece preservado. A execução está desabilitada por padrão.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs/linear_probe").is_dir())
RUN_ABLATION = False
SCENARIOS = ["mono_disc", "mono_cont", "poly_disc", "poly_cont"]
print("Projeto:", ROOT)
print("Treinamento autorizado neste notebook:", RUN_ABLATION)

## Protocolo

TS9 é removido de treino, validação e teste. As gravações restantes mantêm suas partições por fonte (seed 42); os embeddings nativos de 32 kHz são reutilizados após validação. Uma cabeça de 768 → 12 é treinada do início. Busca de oito configurações; duas finalistas com três seeds; cinco seeds novas finais (101, 202, 303, 404, 505). Seleção por F1 macro de validação, AdamW, teto de 200 épocas e parada antecipada. Sem fine-tuning. O teste já foi consultado; esta é uma ablação exploratória.

In [ ]:
if RUN_ABLATION:
    subprocess.run([sys.executable, str(ROOT / "scripts/run_transfer_no_ts9.py"), "--run", "--scenarios", *SCENARIOS], cwd=ROOT, check=True)
else:
    print("Modo consulta: nenhum treinamento ou extração será iniciado.")

In [ ]:
summary_path = ROOT / "results/passt_htsat_transfer/transfer_learning_no_ts9_summary.json"
if not summary_path.exists():
    raise FileNotFoundError("O relatório da ablação ainda não está disponível; abrir este notebook não inicia treinamento.")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
rows = []
for scenario, item in summary["scenarios"].items():
    for model, comparison in item["comparison"].items():
        result = comparison["new_12_class_head"]
        rows.append({"Cenário": scenario, "Modelo": model, **{key: f"{100*result[key]['mean']:.2f} ± {100*result[key]['std']:.2f}" for key in ("accuracy", "macro_f1", "macro_precision", "macro_recall")}})
display(pd.DataFrame(rows))

In [ ]:
comparisons = []
for scenario, item in summary["scenarios"].items():
    for model, values in item["comparison"].items():
        comparisons.append({"Cenário": scenario, "Modelo": model, "Acurácia original (13 classes)": 100*values["original_13_classes_full_test"]["accuracy"]["mean"], "Cabeça antiga nos mesmos WAVs": 100*values["original_13_output_head_on_same_12_class_wavs"]["accuracy"]["mean"], "Cabeça nova sem TS9": 100*values["new_12_class_head"]["accuracy"]["mean"]})
display(pd.DataFrame(comparisons).round(2))

In [ ]:
display(Markdown((ROOT / "results/passt_htsat_transfer/transfer_learning_no_ts9_report.md").read_text(encoding="utf-8")))

## Artefatos e inferência

Os relatórios e JSONs são portáteis. Caches, cabeças, scalers, predições, curvas e logs ficam locais em `results/transfer_learning/ablations/no_ts9/`. Para inferência, usar `src.models.consolidated_probe.load_consolidated_probe` com o diretório da seed e carregar obrigatoriamente o scaler; consultar `probe.classes` para interpretar as 12 saídas. Nenhum arquivo TS9 foi apagado.